# SAADE–AUA: повний експеримент з виявлення синтетичного мовлення

**Scenario-Adaptive Audio Deepfake Evaluation (SAADE)** — сценарно-адаптивне оцінювання стійкості real/fake-аудіодетекторів на українському наборі AUA.

**Послідовність:** каталог і перевірка походження → вибірка → групово-незалежний train/validation/hidden_test → заморожування split → E1 (звичайне навчання) → аналіз слабких акустичних сценаріїв → E2 (random augmentation) і E3 (adaptive augmentation) → фіксація протоколу → фінальний hidden test → EER/ROC-AUC/BA/ASRI → кластерний bootstrap, GEE, MCDM → графіки та ZIP.

**Факти з аудиту AUA за 14.11.2026:** `/home/jovyan/work/AUA`; 112 219 файлів загалом, але канонічне аудіо — **12,665 год** (`own_processed` 2,045, `external_selected` 5,500, `synthetic` 5,119). Оригінали `own_master`, копії у `05_splits`, відео і тексти **не зараховуються повторно**. У самому ZIP із чату містяться **звіти, а не аудіофайли**; медіа повинні залишатися на сервері.

Методологічна основа: [MAKE 2026, DOI 10.3390/make8060161](https://doi.org/10.3390/make8060161). Офіційні антиспуфінгові архітектури: [AASIST](https://github.com/clovaai/aasist), [ASVspoof5 RawNet2](https://github.com/asvspoof-challenge/asvspoof5). ASRI — **дослідницька пропозиція**, не загальноприйнятий стандарт. Еталонні EER/ROC-AUC звітуються обов'язково.

> Безпека методології: мітки папок перевіряються проти наявного metadata `mode`. Спотворення зберігають мітку реальне/синтетичне. Один мовець або похідні одного оригіналу не повинні розходитись між split. **Hidden test не використовується для підбору сценаріїв.** Повторні запуски не редагують AUA: результати зберігаються лише у `cfg.out`.

## Етап 0. Середовище і параметри

Відкрийте ноутбук поруч із `saade_core.py`. Для першого запуску виберіть малу вибірку й 1–2 епохи, потім збільшіть параметри. Для публікації використовуйте `allow_weak_groups=False` та перевірені джерела/ліцензії. Якщо Jupyter не бачить `ffmpeg`, необхідно встановити його в операційній системі сервера.

Залежності: `numpy`, `pandas`, `scipy`, `soundfile`, `scikit-learn`, `joblib`, `matplotlib`, `statsmodels`, `torch`. **Не перевстановлюйте PyTorch без перевірки сумісності з CUDA.**

In [ ]:
from pathlib import Path
import os, sys, platform, shutil, subprocess
import numpy as np
import pandas as pd
from IPython.display import display

HERE = Path.cwd()
sys.path.insert(0, str(HERE))
try:
    import saade_core as S
except ImportError as exc:
    raise ImportError("Покладіть saade_core.py поруч із цим ноутбуком. Завантажте ZIP проєкту та розпакуйте файли в одну папку JupyterLab.") from exc

# ЗМІНИТИ ЛИШЕ ЗА ПОТРЕБИ:
DATA_ROOT = Path('/home/jovyan/work/AUA')
if not (DATA_ROOT / '01_audio').exists():
    candidates=[p for p in [HERE, *HERE.parents, Path('/home/jovyan/work'),Path.home()] if (p/'AUA'/'01_audio').exists()]
    if candidates: DATA_ROOT=candidates[0]/'AUA'

cfg = S.Config(
    root=str(DATA_ROOT),
    out='/home/jovyan/work/results/SAADE_AUA_2026_v1',
    seed=42, seconds=4.0, sample_rate=16000,
    max_items=400,  # 0 = усі придатні записи (потребує більше GPU/часу)
    epochs=4, batch_size=12, num_workers=0,
    allow_weak_groups=False,
    # Стартовий набір повністю реалізованих baseline-моделей:
    models=('spectrogram_cnn','raw_wave_cnn','mfcc_svm'),
    # Пізніше можна замінити на ('aasist','rawnet2','wavlm'),
    # але для них потрібні офіційні репозиторії / pretrained WavLM.
    bootstraps=200,
    test_finalization=False,
)
OUT=Path(cfg.out);OUT.mkdir(parents=True,exist_ok=True)
print('AUA:', cfg.root, 'exists:', (Path(cfg.root)/'01_audio').is_dir())
print('Результати:',OUT)
print('Python:',platform.python_version(),'ffmpeg:',shutil.which('ffmpeg'))
try:
    import torch
    print('PyTorch:',torch.__version__,'GPU:',torch.cuda.is_available(),
          'GPU name:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
except ImportError as e:
    raise ImportError('Встановіть PyTorch, сумісний із CUDA на вашому сервері.') from e

### 0.1. Перевірка необхідних пакетів і кодеків

Без `ffmpeg` сценарії **справжньої MP3-компресії** не працюватимуть; ми не замінюємо їх фіктивною апроксимацією. Якщо у вас немає `soundfile` або інших пакетів, встановіть їх у терміналі або в окремій комірці `%pip install ...`.

In [ ]:
import importlib.util
required=['numpy','pandas','scipy','soundfile','sklearn','joblib','matplotlib','statsmodels','torch']
missing=[p for p in required if importlib.util.find_spec(p) is None]
print('Відсутні пакети:',missing or 'немає')
print('ffmpeg:', 'OK' if shutil.which('ffmpeg') else 'НЕ ЗНАЙДЕНО — встановіть системний ffmpeg')
if missing: raise RuntimeError('Встановіть відсутні пакети й перезапустіть kernel')
if not shutil.which('ffmpeg'): raise RuntimeError('ffmpeg потрібен для реальних MP3-сценаріїв.')

## Етап 1. Виявлення AUA та очищення каталогу

Читаємо **реальні** метадані `01_audio/metadata_audio.csv` (поля `source_dataset`, `speaker_id`, `source_record_id`, `mode`, `license`) та доповнення `own_processed/metadata_audio.csv`, `own_master/metadata_audio.csv`, `synthetic/new-tts/metadata_new-tts.csv`. Відсікаємо пошкоджені записи, явні конфлікти міток і короткі аудіо. Список усіх проблем — у `01_catalog_issues.csv`.

**Зверніть увагу:** мітка з каталогу `synthetic` — це припущення на основі шляху, поки метадані або експерт не підтвердили її. Такі джерела потребують окремої перевірки перед публікацією.

In [ ]:
catalog=S.build_catalog(cfg)
display(pd.read_csv(OUT/'01_sources_summary.csv').head(20))
print('Мітки, визначені лише з папки:', int((catalog.label_source=='folder_inferred').sum()))
print('Конфліктні мітки:', int(catalog.label_conflict.sum()))
print('Слабкі групи:', int((catalog.group_confidence=='weak_file_fallback').sum()))
display(catalog[catalog.warning.ne('')][['relative_path','label','group_confidence','warning']].head(15))

## Етап 2. Збалансована експериментальна вибірка

Вибираємо до 200 real та 200 fake, обмежуючи надмірне домінування однієї групи. Це не означає, що 400 незалежних мовців існують: їхня кількість визначається за наявними метаданими. Якщо після очищення менше ніж 400 придатних файлів, працюємо з фактичним числом і документуємо зменшення.

In [ ]:
subset=S.get_experimental_subset(catalog,cfg)
display(subset.groupby(['label','source']).agg(files=('path','size'),groups=('group_id','nunique'),hours=('duration_sec',lambda x:x.sum()/3600)).reset_index())

## Етап 3. Нові незалежні split і захист від витоків

Не використовуємо готові `05_splits` без перевірки: у попередніх аудитах траплялися неповні зв'язки master→variant. Код формує **нові** train/validation/hidden_test із груповою ізоляцією. Якщо бракує `speaker_id` або надійного `source_record_id`, код зупиняється. Для технічної проби можна поставити `cfg.allow_weak_groups=True`, але цей режим не придатний для доведених claims про відсутність витоків. Заморожуємо протокол до навчання.

In [ ]:
splits=S.make_splits(subset,cfg)
S.lock_hidden_test(splits,cfg)
repeated_text=S.transcript_leakage_report(splits,cfg)
gen_holdout=S.generator_holdout_report(splits,cfg)
print('Підозрілі однакові транскрипти між split:',len(repeated_text))
display(pd.read_csv(OUT/'03_split_summary.csv'))
display(gen_holdout.head(20))

## Етап 4. Генератор сценаріїв — перевірка одного аудіо

Для кожного файлу матимемо 13 умов: початкова + по три рівні шуму, MP3-компресії, синтетичної реверберації та lowpass. **Вихідна мітка real/fake не змінюється.** Для статті потрібно вказувати, що реверберація генерується синтетично, а не через виміряні RIR.

In [ ]:
scenarios_val=S.scenario_manifest(splits,split='validation')
print('Валідаційних сценаріїв:',len(scenarios_val),'(для',len(splits[splits.split=='validation']),'базових файлів)')
display(scenarios_val[['relative_path','category','level','value']].head(15))
example=scenarios_val.iloc[0]
wave=S.read_wave(example.path,cfg.sample_rate,cfg.seconds,cfg.seed)
print('1 файл:',len(wave),'samples; first 10 samples:',wave[:10].round(3))

## Етап 5. E1 — початкова модель без аугментацій

Навчаємо три архітектурно відмінні baselines: `spectrogram_cnn`, `raw_wave_cnn`, `mfcc_svm`. Це **не** AASIST/RawNet2. Опціональні офіційні архітектури можна підключити через параметр `cfg.models`, як описано в кінці. Усі початкові метрики й пороги обчислюються **виключно на validation**, прихований тест не відкривається. Якщо модель навчена раніше з тими самими параметрами, завантажується checkpoint із `OUT/models`.

In [ ]:
models={}; model_info={}; baseline_val_predictions=[]
for name in cfg.models:
    print('\n=== BASELINE:',name,'===')
    m,meta=S.train_one(splits,cfg,name,regime='static')
    models[(name,'static')]=m;model_info[(name,'static')]=meta
    fname=OUT/f'04_val_predictions_{S.safe_key(name)}_static.csv'
    if fname.exists():
        p=pd.read_csv(fname)
        if len(p)!=len(scenarios_val) or set(p.scenario_id)!=set(scenarios_val.scenario_id):
            raise RuntimeError(f'Cached scenarios mismatch for {fname}; use a NEW experiment output directory')
    else:
        p=S.run_scenario_predictions(splits,m,name,'static',cfg,
            threshold=meta['threshold'],split='validation')
        p.to_csv(fname,index=False)
    baseline_val_predictions.append(p)
base_val=pd.concat(baseline_val_predictions,ignore_index=True)
print('Baseline scenario rows:',len(base_val))
display(base_val.head())

## Етап 6. Визначення слабких сценаріїв — адаптивний цикл

На основі E1 визначаємо категорії з найбільшою часткою помилок, усереднюючи **окремо за real/fake і рівнями складності**. Це не використовує hidden test. Для стійкого порівняння залишаємо ненульову ймовірність для всіх категорій (параметр згладжування).

In [ ]:
weights_by_model={name:S.validation_difficulty(base_val[base_val.model==name]) for name in cfg.models}
weights={c:float(np.mean([weights_by_model[n][c] for n in cfg.models])) for c in S.CATEGORIES}
total=sum(weights.values());weights={c:v/total for c,v in weights.items()}
S.save_json(OUT/'05_adaptive_weights.json',dict(weights=weights,by_model=weights_by_model,
    source='validation only; E1 static detectors'))
print('Адаптивні ймовірності:',weights)
display(pd.DataFrame([weights]).T.rename(columns={0:'probability'}))

## Етап 7. E2/E3 — random проти adaptive аугментацій

Заново навчаємо кожну модель з однаковим seed, бюджетом епох і часткою аугментованих прикладів (`augmentation_probability`). В E2 категорія випадкова з рівномірним розподілом. В E3 категорія вибирається на основі **валидаційної** вразливості E1. Створювати всі варіанти на диску не потрібно: акустичне перетворення виконується під час навчання.

Це порівняння оцінює насамперед **користь адаптивного вибору сценаріїв для стійкості детектора**. Якщо стаття присвячується лише аудиту моделей без донавчання, E2/E3 можна подати як додатковий експеримент.

In [ ]:
all_val=[base_val]
for name in cfg.models:
    for regime in ('random','adaptive'):
        print('\n===',name,regime,'===')
        m,meta=S.train_one(splits,cfg,name,regime=regime,
            adaptive_weights=weights if regime=='adaptive' else None)
        models[(name,regime)]=m;model_info[(name,regime)]=meta
        f=OUT/f'04_val_predictions_{S.safe_key(name)}_{regime}.csv'
        if f.exists():
            p=pd.read_csv(f)
            if len(p)!=len(scenarios_val) or set(p.scenario_id)!=set(scenarios_val.scenario_id):raise RuntimeError('Cached validation scenario set differs')
        else:
            p=S.run_scenario_predictions(splits,m,name,regime,cfg,
                threshold=meta['threshold'],split='validation')
            p.to_csv(f,index=False)
        all_val.append(p)
validation_predictions=pd.concat(all_val,ignore_index=True)
validation_predictions.to_csv(OUT/'04_val_predictions_ALL.csv',index=False)
print('Validation scored:',len(validation_predictions))

## Етап 8. Порівняння E1/E2/E3 на validation

Виводимо EER, ROC-AUC, Balanced Accuracy, F1, помилки real/fake та ASRI (лише 12 акустичних умов, без baseline). Ці результати застосовуються для вибору заздалегідь зафіксованого протоколу. Кількість **сценаріїв** не дорівнює кількості незалежних учасників; у статистиці bootstrap потрібно ресемплювати групи, а не файли-сценарії.

In [ ]:
val_metric,val_asri=S.summarize_predictions(validation_predictions,cfg)
print('Validation ASRI:')
display(val_asri.sort_values(['model','regime']))
print('Validation mean per-category EER & BA:')
display(val_metric[val_metric.category!='baseline'].groupby(['model','regime']).agg(
    avg_EER=('eer','mean'),avg_BA=('balanced_accuracy','mean'),avg_AUC=('roc_auc','mean'),
    worst_BA=('balanced_accuracy','min')).reset_index())
S.render_charts(val_metric,val_asri,cfg)

## Етап 9. Фіксація всього експериментального протоколу

Після перевірки validation обов'язково фіксуємо checkpoint SHA-256, пороги та split SHA-256. **Не змінюйте** `max_items`, `seed`, архітектури, параметри або аудіофайли після цього в межах того самого `OUT`: для нового експерименту створіть новий каталог результатів.

In [ ]:
frozen=S.save_locked_protocol(cfg,splits,model_info)
print('Збережено 06_FINAL_PROTOCOL.json; checkpoints:',len(frozen['models']))
print('Split SHA:',frozen['dataset_hash'])

## Етап 10. ФІНАЛЬНИЙ hidden test — однократна перевірка

**Спочатку завершіть всі попередні етапи.** За замовчуванням цей блок пропускає hidden test. Для фінальної оцінки змініть лише `RUN_FINAL_HIDDEN_TEST=True` і запустіть цю клітинку. Зберігайте отриманий набір результатів, не змінюйте ваги й сценарії після перегляду hidden test. Якщо експеримент потрібно переробити, створіть новий незалежний test або позначте наступну оцінку як exploratory.

In [ ]:
RUN_FINAL_HIDDEN_TEST = False  # ВСТАНОВІТЬ True ЛИШЕ ПІСЛЯ ФІКСАЦІЇ МОДЕЛЕЙ
if RUN_FINAL_HIDDEN_TEST:
    cfg.test_finalization=True
    S.ensure_test_gate(cfg)
    S.verify_locked_protocol(cfg,splits)
    test_parts=[]
    for (name,regime),model in models.items():
        print('\nHIDDEN TEST:',name,regime)
        f=OUT/f'07_hidden_{S.safe_key(name)}_{S.safe_key(regime)}.csv'
        if f.exists():p=pd.read_csv(f)
        else:
            p=S.run_scenario_predictions(splits,model,name,regime,cfg,
                 threshold=model_info[(name,regime)]['threshold'],split='hidden_test')
            p.to_csv(f,index=False)
        test_parts.append(p)
    hidden_predictions=pd.concat(test_parts,ignore_index=True)
    hidden_predictions.to_csv(OUT/'07_hidden_predictions_ALL.csv',index=False)
    hidden_metrics,hidden_asri=S.summarize_predictions(hidden_predictions,cfg)
    hidden_metrics.to_csv(OUT/'08_HIDDEN_scenario_metrics.csv',index=False)
    hidden_asri.to_csv(OUT/'09_HIDDEN_ASRI.csv',index=False)
    print('FINAL HIDDEN RESULTS:')
    display(hidden_asri)
else:
    print('Hidden test залишається ЗАКРИТИМ. Спочатку перевірте validation, зафіксуйте моделі; для фінального запуску поставте True.')

## Етап 11. Статистика, експертні звіти й графіки

Якщо фінальний hidden test уже проведений, використовуйте тільки його для кінцевих висновків, 95% довірчих інтервалів і парного E2–E3 порівняння; інакше формується лише **validation-аналіз**. Допоміжний GEE вивчає фактори вразливості; MCDM/SRD/TOPSIS — **розвідувальне ранжування**, не замінює статистичного тесту.

In [ ]:
pred_file=OUT/'07_hidden_predictions_ALL.csv'
if pred_file.exists() and RUN_FINAL_HIDDEN_TEST:
    analyzed=pd.read_csv(pred_file)
    analysis_type='HIDDEN FINAL'
else:
    analyzed=validation_predictions.copy()
    analysis_type='VALIDATION EXPLORATORY'
print('Статистичний аналіз:',analysis_type,'| рядків:',len(analyzed))
metrics,ari=S.summarize_predictions(analyzed,cfg)
stratified=S.summarize_source_results(analyzed,cfg)
boot=[]
for name in cfg.models:
    result=S.paired_group_bootstrap_delta(analyzed,name,B=cfg.bootstraps,seed=cfg.seed)
    result['analysis_type']=analysis_type
    boot.append(result)
boot_df=pd.DataFrame(boot);boot_df.to_csv(OUT/'11_bootstrap.csv',index=False)
display(boot_df)
ranking=S.mcdm_rank(metrics,ari,cfg)
display(ranking[['model','regime','ASRI','mean_EER','mean_AUC','MCDM_consensus']] if not ranking.empty else ranking)
_gee=S.gee_cluster_effects(analyzed,cfg)
S.render_charts(metrics,ari,cfg)
print('Source-stratified metrics:');display(stratified.head(20))

## Етап 12. Пакування результатів для статті

ZIP містить **тільки** CSV, JSON, PNG, TXT і Markdown. Не включає аудіофайли, персональні відомості з документів згоди та `.env` із токенами. Усі методичні застереження записуються в `README_RESULTS.md`.

In [ ]:
readme=S.make_readme(cfg,splits)
print(readme)
archive=S.zip_results(cfg,include_models=False)
print('Завантажте ZIP через браузер файлів JupyterLab і надішліть його для аналізу.')

---
## Додатковий етап A. Офіційні AASIST, RawNet2 і WavLM

**У цьому проєкті вони справжні, а не спрощені заміни.** Для AASIST і RawNet2 потрібно завантажити відповідні офіційні репозитарії на сервер (Git/GPU), а для WavLM — встановити `transformers` та надати доступ до ваг `microsoft/wavlm-base-plus`.

У терміналі JupyterLab (один раз):

```bash
cd /home/jovyan/work
git clone https://github.com/clovaai/aasist.git
git clone https://github.com/asvspoof-challenge/asvspoof5.git
# python -m pip install transformers   # лише якщо потрібен WavLM
```

Після встановлення задайте у комірці конфігурації: `cfg.models=('aasist','rawnet2','wavlm')`, переконайтесь, що `cfg.repo_aasist`, `cfg.repo_asvspoof5` вказують на репозиторії, та запустіть **у новому** `cfg.out`. Для кожної архітектури застосовується той самий протокол, але початкові параметри тренування можуть потребувати додаткового налаштування в межах validation. **Не завантажуйте випадковий pretrained checkpoint з інтернету без перевірки структури шарів і орієнтації міток.**

Офіційні посилання: [AASIST](https://github.com/clovaai/aasist), [ASVspoof5 baselines](https://github.com/asvspoof-challenge/asvspoof5), [WavLM](https://huggingface.co/microsoft/wavlm-base-plus).

## Додатковий етап B. Невідомий TTS-генератор

Файл `03_generator_holdout_feasibility.csv` перелічує генератори, які потенційно можна вилучити з навчання і протестувати окремо. Для досліду leave-one-generator-out **треба заново сформувати train/validation, виключити всі приклади відповідного генератора та пов'язані group_id**, і лише потім оцінювати на не баченому генераторі. Просте порівняння source-stratified test без такого вилучення не дорівнює zero-shot генераторному переносу.

## Обмеження щодо регіональності

Текст із регіональними словами не підтверджує походження чи акцент мовця. Дані з `script_profile` без підтвердженого `speaker_origin_group` не можна називати аудіо певного діалекту.